# MedSAM 3 on PadChest-GR — uncertainty-stratified IoU eval

End-to-end Colab notebook. For every (image, grounded finding) pair in
PadChest-GR we:

1. clean the radiologist's finding sentence (strip hedging language) so the
   text prompt going into MedSAM 3 is unbiased,
2. run MedSAM 3 with the cleaned sentence as a text prompt to get a
   predicted segmentation mask,
3. compute a bundle of IoUs against the ground-truth annotator boxes (per
   box, intersection, pixel-OR union, outer-bbox union, pairwise
   inter-annotator agreement, reader-vs-reader),
4. aggregate the bundle stratified by the rule-based spatial-uncertainty
   label from `rule_uncertainty_scores.jsonl`.

Inputs:
* `gs://${BUCKET}/images/{image_id}` — extracted PadChest-GR PNGs.
* `project/data/processed/samples_with_uncertainty_and_iou.csv` — sample
  list with reader1/reader2 boxes, finding label, MedGemma uncertainty
  label.
* `project/data/processed/rule_uncertainty_scores.jsonl` — rule-based
  uncertainty label (used for stratification + for hedge stripping).
* `lal-Joey/MedSAM3_v1` — LoRA weights on top of `facebook/sam3`.

Outputs (per sample, written to GCS):
* `gs://${BUCKET}/outputs/medsam3_v1/{sample_id}.json` — IoU bundle.
* `gs://${BUCKET}/outputs/medsam3_v1/masks/{sample_id}.npz` — predicted binary mask.

Outputs (aggregated, copied back to the repo via Drive):
* `project/outputs/medsam3_per_sample_ious.csv`
* `project/outputs/medsam3_aggregate_ious.json`
* `project/outputs/medsam3_per_finding_ious.csv`
* `project/figures/medsam3_iou_by_uncertainty_group.png`
* `project/figures/medsam3_iou_histogram_by_group.png`
* `project/figures/medsam3_agreement_vs_iou_scatter.png`


## Cell 0 — one-time GCS extraction of the split-zip PadChest-GR archive

The dataset was uploaded as `gs://${BUCKET}/Padchest_GR_files/PadChest_GR.zip.001..037`
(byte-split, ~1 GB chunks). For streaming we need loose PNGs at
`gs://${BUCKET}/images/{image_id}`. This cell is idempotent: if
`gs://${BUCKET}/images/` is already populated, it no-ops.

Disk math on Colab: 36 GB zips + 36 GB extracted ≈ 72 GB of 78 GB free
runtime disk. Tight on Colab Free; comfortable on Colab Pro. The cell
streams chunks → concatenated zip → `unzip` directly into a folder, then
deletes each piece as it goes so peak disk stays bounded.

If you run this cell more than once it just checks the destination and
exits.

In [ ]:
# Cell 0 uses the modern `gcloud storage` CLI, NOT the legacy `gsutil`.
# Background: in Colab, auth.authenticate_user() seeds Application Default
# Credentials (ADC) which the Python SDK and `gcloud storage` both respect,
# but the older `gsutil` binary on the Colab image silently ignores ADC and
# falls back to anonymous access → 401 "Anonymous caller". Google's own
# error message says "use gcloud storage instead of gsutil", which is what
# we do here.

BUCKET      = "yang-padchest-gr"          # change if your bucket name differs
GCS_PROJECT = "yang-uncertainty-eval"     # your GCP project id
SRC_PREFIX  = "Padchest_GR_files"
DST_PREFIX  = "images"

# --- Auth FIRST -----------------------------------------------------------
from google.colab import auth
auth.authenticate_user()
get_ipython().system(f"gcloud config set project {GCS_PROJECT} -q")
print("Active identity:")
get_ipython().system("gcloud auth list --filter=status:ACTIVE --format='value(account)'")

# --- Preflight: confirm we can actually read the bucket -------------------
# (`gcloud storage ls` is the modern, ADC-respecting replacement for
# `gsutil ls`. If THIS fails the rest of the cell can't possibly succeed.)
import subprocess
r = subprocess.run(
    f"gcloud storage ls gs://{BUCKET}/ 2>&1 | head -10",
    capture_output=True, text=True, shell=True,
)
print(f"\nPreflight `gcloud storage ls gs://{BUCKET}/`:")
print(r.stdout)
if "ERROR" in r.stdout or "403" in r.stdout or "404" in r.stdout or "401" in r.stdout:
    raise RuntimeError(
        f"Cannot read gs://{BUCKET}/ as the active account. Verify:\n"
        f"  - bucket name '{BUCKET}' is exactly right\n"
        f"  - the auth popup signed in as the account that owns the bucket\n"
        f"  - the bucket is in project '{GCS_PROJECT}'\n"
    )

# --- Helpers (using google-cloud-storage SDK, which uses ADC reliably) ---
from google.cloud import storage
_client = storage.Client(project=GCS_PROJECT)
_bucket = _client.bucket(BUCKET)

def gcs_has_images():
    blobs = list(_bucket.list_blobs(prefix=f"{DST_PREFIX}/", max_results=1))
    return len(blobs) > 0

# --- Main extraction flow -------------------------------------------------
if gcs_has_images():
    print(f"\ngs://{BUCKET}/{DST_PREFIX}/ already populated — skipping extraction.")
else:
    print("\nExtracting PadChest-GR from split-zip parts in the bucket. One-time slow step.")
    !mkdir -p /content/zips /content/images /content/extracted
    # 1) Pull split-zip parts via gcloud storage (uses ADC, no 401)
    get_ipython().system(
        f"gcloud storage cp 'gs://{BUCKET}/{SRC_PREFIX}/PadChest_GR.zip.*' /content/zips/"
    )
    # 2) Concatenate to a single zip, then delete the parts to save disk
    !ls /content/zips/PadChest_GR.zip.* | sort > /content/zip_parts.txt
    !cat $(cat /content/zip_parts.txt | tr '\n' ' ') > /content/full.zip
    !rm -f /content/zips/PadChest_GR.zip.*
    # 3) Unzip into /content/extracted/
    !unzip -q -o /content/full.zip -d /content/extracted/
    !rm -f /content/full.zip
    # 4) Flatten into /content/images/{filename.png}
    !find /content/extracted -type f -name '*.png' -print0 | xargs -0 -I{} cp {} /content/images/
    # 5) Push to gs://BUCKET/images/ via gcloud storage
    get_ipython().system(
        f"gcloud storage rsync /content/images/ gs://{BUCKET}/{DST_PREFIX}/ --recursive"
    )
    !rm -rf /content/extracted /content/images
    print("Extraction + upload complete.")


## Cell 1 — install dependencies (Colab)

Pins PyTorch to a CUDA-12.6 wheel because MedSAM 3 / SAM 3 require it; Colab's
stock torch is older. Installs the MedSAM 3 repo in editable mode so the
`inference.py` helpers `load_model_with_lora` / `segment_image` are importable.


In [ ]:
# CUDA-12.6 PyTorch wheels for SAM 3.
# IMPORTANT: pin torch + torchvision + torchaudio TOGETHER. If only some of
# them are upgraded, transformers' lazy-import chain for Sam3Model trips on
# a CUDA-version mismatch and surfaces a misleading
# "Could not import module 'Sam3Model'" error.
#
# We use --force-reinstall because pip's "--upgrade" silently skips a
# package if the PyPI version number is higher than the alternate-index
# wheel — which is the case for torchaudio (PyPI cu128 build vs our cu126
# wheel). Without --force-reinstall pip leaves torchaudio at cu128 and
# you get the misleading Sam3Model error.
!pip -q install --force-reinstall --no-deps \
    torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu126


# MedSAM 3 (LoRA fine-tune of facebook/sam3) — has the inference helpers.
import os
if not os.path.isdir('/content/medsam3'):
    !git clone https://github.com/Joey-S-Liu/MedSAM3.git /content/medsam3
%cd /content/medsam3
!pip -q install -e .

# Transformers needs to be recent enough to have Sam3Model. (The Colab
# default is usually fine after `pip install -e .` above pulls its
# requirement; this line just guarantees a known-good floor.)
!pip -q install --upgrade 'transformers>=4.50'

# GCS streaming + huggingface
!pip -q install --upgrade gcsfs google-cloud-storage huggingface_hub

# Plotting / utilities (Colab usually has these but pinning won't hurt)
!pip -q install --upgrade scikit-image tqdm pandas matplotlib

print("\n>>> RESTART THE RUNTIME NOW <<<")
print("Runtime → Restart session, then run Cells 2 → 6.")
print("(torch was already loaded into the kernel; pip can't hot-swap it.)")


## Cell 2 — authenticate to Google + Hugging Face

* Google auth lets `gcsfs` and `gsutil` access your private bucket without
  a service-account JSON.
* HF login is needed to download the gated `facebook/sam3` base weights.
  The MedSAM 3 LoRA repo (`lal-Joey/MedSAM3_v1`) is public.


In [ ]:
from google.colab import auth
auth.authenticate_user()

from huggingface_hub import notebook_login
notebook_login()


## Cell 3 — config

All paths + hyperparameters in one place so you can edit them without
hunting through later cells. `MASK_GRID` matches your repo's
`MASK_GRID_SIZE = 1024` so the predicted-mask IoU is computed on the same
grid as the existing reader-reader IoU.

In [ ]:
GCS_PROJECT = "yang-uncertainty-eval"   # your GCP project id
BUCKET      = "yang-padchest-gr"        # your bucket
IMG_PREFIX  = "images"                  # gs://BUCKET/images/<image_id>
OUT_PREFIX  = "outputs/medsam3_v1"      # per-sample JSON + .npz masks
MASK_PREFIX = f"{OUT_PREFIX}/masks"

# Drive (for pushing final aggregates back to the repo)
DRIVE_REPO_ROOT = "/content/drive/MyDrive/uncertaintyestimate"

# Inference
BASE_MODEL  = "facebook/sam3"
LORA_REPO   = "lal-Joey/MedSAM3_v1"
# Actual filename in the HF repo (verified 2026-06: `best_lora_weights.pt`).
# If the maintainer renames/republishes it, update this string after running
# `huggingface_hub.list_repo_files('lal-Joey/MedSAM3_v1')`.
LORA_WEIGHTS_FILE = "best_lora_weights.pt"

# Prompt source for MedSAM 3:
#   "rule"     — deterministic regex strip of UNCERTAIN_TERMS (fast, lossy)
#   "medgemma" — MedGemma 4B rewrites every sentence into a concise noun
#                phrase (slow upfront, cleaner prompts — preferred)
#   "raw"      — pass the original sentence verbatim (worst, for ablation)
MEDSAM3_PROMPT_SOURCE = "medgemma"

# MedGemma rewriter (only used if MEDSAM3_PROMPT_SOURCE == "medgemma")
# Matches what the rest of the repo uses for uncertainty classification.
MEDGEMMA_MODEL_NAME    = "google/medgemma-4b-it"
MEDGEMMA_REWRITE_CACHE = "/content/medsam3_prompts_medgemma.csv"
MEDGEMMA_MAX_NEW_TOK   = 40   # generous for a 3-10 word phrase

MASK_GRID   = 1024
MASK_THRESHOLD = 0.5


# Smoke-test
SMOKE_N = 50          # how many samples to run in the smoke cell
SMOKE_SEED = 42       # fixed so the smoke set is reproducible

# Where the source label files live (mounted via Drive in Cell 4)
SAMPLES_CSV  = f"{DRIVE_REPO_ROOT}/project/data/processed/samples_with_uncertainty_and_iou.csv"
RULE_JSONL   = f"{DRIVE_REPO_ROOT}/project/data/processed/rule_uncertainty_scores.jsonl"

print("Config loaded.")


## Cell 4 — load samples + build the cleaned MedSAM 3 prompt

1. Fetch the two small label files (≈3 MB total) **directly from GitHub**
   — no Drive copy needed. They live under `project/data/processed/` in
   the repo and rarely change.
2. Mount Drive **only** so Cell 12 has a place to push results back to
   the repo at the end. If you skip this mount, Cells 0–11 still work;
   Cell 12 will just write the aggregates to `/content/` instead.
3. Load `samples_with_uncertainty_and_iou.csv` — `image_id`, `sample_id`,
   `sentence`, `finding_label`, `reader1_boxes`, `reader2_boxes`, plus the
   MedGemma `uncertainty_label`.
4. Load `rule_uncertainty_scores.jsonl` — same shape, but rule-based label.
   We **stratify by this one**. MedGemma label is kept as an ablation column.
5. Define `clean_sentence_rule()` — case-insensitive strip of every term in
   `UNCERTAIN_TERMS`. This is the prompt MedSAM 3 actually sees.


In [ ]:
# --- where the label files come from --------------------------------------
# Public GitHub URL of the current branch. If you fork the repo, change
# REPO_RAW_URL to point at your fork. Files are tiny (~3 MB total) so
# downloading them is faster than mounting Drive.
REPO_RAW_URL = "https://raw.githubusercontent.com/nprakash1/uncertaintyestimateyang/rule-bag-of-words"
SAMPLES_CSV  = "/content/samples_with_uncertainty_and_iou.csv"
RULE_JSONL   = "/content/rule_uncertainty_scores.jsonl"

import os
if not os.path.exists(SAMPLES_CSV):
    get_ipython().system(
        f"curl -fsSL '{REPO_RAW_URL}/project/data/processed/samples_with_uncertainty_and_iou.csv' "
        f"-o '{SAMPLES_CSV}'"
    )
if not os.path.exists(RULE_JSONL):
    get_ipython().system(
        f"curl -fsSL '{REPO_RAW_URL}/project/data/processed/rule_uncertainty_scores.jsonl' "
        f"-o '{RULE_JSONL}'"
    )
print(f"samples CSV: {os.path.getsize(SAMPLES_CSV):,} bytes")
print(f"rule  JSONL: {os.path.getsize(RULE_JSONL):,} bytes")

# --- mount Drive (best-effort, only needed for Cell 12) -------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
except Exception as e:
    print(f"[warn] Drive not mounted ({e}); Cell 12 will fall back to /content/.")
    DRIVE_MOUNTED = False

import json, re, pandas as pd
from pathlib import Path


# Hedging vocab — kept inline so the notebook is self-contained and matches
# project/src/medgemma_uncertainty.py::UNCERTAIN_TERMS exactly. If you edit
# that file, mirror the edit here.
UNCERTAIN_TERMS = [
    # diagnostic hedges
    "possible","possibly","probable","probably","likely","questionable","equivocal",
    "may represent","could represent","may correspond","could correspond","may be",
    "might","appears","apparent","cannot exclude","can't exclude","cannot be excluded",
    "cannot rule out","difficult to exclude","difficult to assess","suspicious for",
    "suspicion of","suggestive of","suggesting","suggests","compatible with",
    "consistent with","rule out","to rule out","vs ","versus",
    # spatial / boundary / visibility hedges
    "ill-defined","ill defined","poorly defined","poorly-defined","ill-circumscribed",
    "ill circumscribed","indistinct","vague","subtle","faint","hazy","blurred","blurry",
    "fuzzy","obscured","barely visible",
]
# Sort longest-first so multi-word phrases are stripped before their substrings
UNCERTAIN_TERMS = sorted(set(UNCERTAIN_TERMS), key=len, reverse=True)

# Multi-word phrases get matched as case-insensitive substrings; single words
# get a \b...\b boundary so we don't kill "might" inside "mightily".
_PATTERNS = []
for term in UNCERTAIN_TERMS:
    if " " in term or "-" in term:
        _PATTERNS.append(re.compile(re.escape(term), re.IGNORECASE))
    else:
        _PATTERNS.append(re.compile(rf"\b{re.escape(term)}\b", re.IGNORECASE))

def clean_sentence_rule(sentence: str) -> str:
    if not isinstance(sentence, str):
        return ""
    s = sentence
    for p in _PATTERNS:
        s = p.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    # tidy trailing punctuation residue like " ," after a strip in the middle
    s = re.sub(r"\s+([.,;:!?])", r"\1", s).strip(" .,;:")
    return s

# --- load source frames ----------------------------------------------------
samples = pd.read_csv(SAMPLES_CSV)
print(f"Loaded {len(samples)} rows from {SAMPLES_CSV}")
print("Columns:", list(samples.columns))

rule_rows = []
with open(RULE_JSONL, "r") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try:
            rule_rows.append(json.loads(line))
        except Exception:
            continue
rule_df = pd.DataFrame(rule_rows)[["sample_id", "uncertainty_label"]]
rule_df = rule_df.rename(columns={"uncertainty_label": "uncertainty_label_rule"})
print(f"Loaded {len(rule_df)} rule-based labels from {RULE_JSONL}")

# Merge: keep MedGemma label under a clear name; rule label is the strat key.
samples = samples.rename(columns={"uncertainty_label": "uncertainty_label_medgemma"})
samples = samples.merge(rule_df, on="sample_id", how="left")

# Build the rule-based MedSAM 3 prompt (always — used as default + fallback)
samples["medsam3_prompt_rule"] = samples["sentence"].astype(str).map(clean_sentence_rule)

# Drop rows where stripping produced an empty prompt (extremely rare)
empty_after_strip = (samples["medsam3_prompt_rule"].str.len() == 0).sum()
print(f"Rows with empty rule prompt after hedge-strip: {empty_after_strip} (dropping)")
samples = samples[samples["medsam3_prompt_rule"].str.len() > 0].reset_index(drop=True)

# Pick the ACTIVE prompt source (see Cell 3 for the config flag).
# Cell 4b will overwrite `medsam3_prompt` with the MedGemma rewrite if
# MEDSAM3_PROMPT_SOURCE == "medgemma". Until then we default to the rule
# version so the notebook is runnable end-to-end with just Cell 4.
if MEDSAM3_PROMPT_SOURCE == "raw":
    samples["medsam3_prompt"] = samples["sentence"].astype(str)
else:
    samples["medsam3_prompt"] = samples["medsam3_prompt_rule"]

# Stratification summary
print(f"\nActive prompt source (will be overwritten in Cell 4b if 'medgemma'): "
      f"{MEDSAM3_PROMPT_SOURCE}")
print("\nRule-based label distribution:")
print(samples["uncertainty_label_rule"].value_counts(dropna=False))

samples.head(5)[["sample_id","image_id","sentence","medsam3_prompt_rule",
                 "uncertainty_label_rule","uncertainty_label_medgemma"]]


## Cell 4b — MedGemma prompt rewriter (only if `MEDSAM3_PROMPT_SOURCE == "medgemma"`)

**Why this exists**: the rule-based strip in Cell 4 is fast and deterministic,
but it leaves syntactic artifacts because it's pure regex — e.g. "Suggestive
of mild cardiomegaly" → "of mild cardiomegaly" (dangling "of"). SAM 3's CLIP
text encoder was trained on **noun phrases**, not partial sentences, so
prompt quality matters.

**What this cell does**: loads `google/medgemma-4b-it` once, generates a
concise noun-phrase rewrite of every sentence with a strict template
(temperature=0, max_new_tokens=40), caches the results to
`MEDGEMMA_REWRITE_CACHE` so subsequent runs are instant, then **frees
MedGemma from VRAM** before MedSAM 3 loads in Cell 6 (a T4 cannot hold
both 4B + 5B models simultaneously).

**Cost**: ~3–5 min for ~5K samples on a T4 (greedy decoding, batch 1).
Cached afterwards, so Colab session restarts are cheap.

**Prerequisites**: you must have accepted the license at
https://huggingface.co/google/medgemma-4b-it (gated), and run Cell 2
(`notebook_login`).

If you set `MEDSAM3_PROMPT_SOURCE = "rule"` or `"raw"`, this cell no-ops.


In [ ]:
import os, gc, json, time
import pandas as pd

# Strict template — output rules engineered so MedGemma produces a clean
# noun phrase suitable for SAM 3's CLIP text encoder. Edit carefully.
# (Single-triple-quote is used here to avoid colliding with the outer raw
#  string that wraps this whole cell.)
MEDGEMMA_REWRITE_TEMPLATE = '''You are a medical NLP assistant. Convert ONE radiology finding sentence into a concise lesion description that a text-to-mask image-segmentation model can use as a prompt.


Output rules — follow EXACTLY:
1. Output ONLY the rewritten phrase. No JSON, no explanation, no quotes, no leading/trailing punctuation.
2. 3 to 10 words.
3. Noun phrase form. NO main verb. NO words like "There is", "noted", "seen", "demonstrates".
4. Keep the anatomical location if the original mentions one (e.g., "right lower lobe", "left hilum").
5. Strip every diagnostic hedge: possible, possibly, probable, probably, likely, may, might, suggestive, suggests, consistent with, compatible with, suspicious for, equivocal, questionable, appears, apparent, cannot exclude.
6. Strip every boundary/visibility hedge: ill-defined, indistinct, subtle, faint, hazy, vague, blurry, obscured, barely visible.
7. Use the standard radiology term (e.g., "consolidation", "pleural effusion", "atelectasis", "opacity", "nodule", "mass", "cardiomegaly", "pneumothorax").
8. Preserve laterality (left/right/bilateral) if specified.

Examples:
Sentence: "Subtle, ill-defined opacity in the right lower lobe, possible early consolidation"
Rewrite: right lower lobe consolidation

Sentence: "Suggestive of mild cardiomegaly"
Rewrite: cardiomegaly

Sentence: "There is a small right pleural effusion"
Rewrite: right pleural effusion

Sentence: "Indistinct nodular opacity in the left upper lobe, cannot exclude malignancy"
Rewrite: left upper lobe nodule

Sentence: {sentence}
Rewrite:'''


def _clean_rewrite_output(raw: str) -> str:
    # Take MedGemma's raw text and enforce the output contract.
    if not raw: return ""

    # First line only (in case the model added explanation)
    s = raw.strip().split("\n")[0]
    # Strip wrapping quotes, leading "Rewrite:" if echoed, trailing punctuation
    s = s.strip().strip('"').strip("'")
    if s.lower().startswith("rewrite:"):
        s = s[len("rewrite:"):].strip()
    s = s.rstrip(".,;:! ").strip()
    # Hard cap at 12 words (defensive — template asks for 3-10)
    parts = s.split()
    if len(parts) > 12:
        s = " ".join(parts[:12])
    return s


if MEDSAM3_PROMPT_SOURCE != "medgemma":
    print(f"MEDSAM3_PROMPT_SOURCE = {MEDSAM3_PROMPT_SOURCE!r} — skipping MedGemma rewriter.")
else:
    # --- load cache ------------------------------------------------------
    if os.path.exists(MEDGEMMA_REWRITE_CACHE):
        cached_df = pd.read_csv(MEDGEMMA_REWRITE_CACHE)
        cached_ids = set(cached_df["sample_id"])
        print(f"Loaded {len(cached_df)} cached MedGemma rewrites from {MEDGEMMA_REWRITE_CACHE}")
    else:
        cached_df = pd.DataFrame(columns=["sample_id","medsam3_prompt_medgemma"])
        cached_ids = set()
        print(f"No cache yet at {MEDGEMMA_REWRITE_CACHE}; computing fresh.")

    todo = samples[~samples["sample_id"].isin(cached_ids)][["sample_id","sentence"]].reset_index(drop=True)
    print(f"To rewrite this session: {len(todo)} of {len(samples)} samples")

    if len(todo) > 0:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        from tqdm.auto import tqdm

        print(f"Loading {MEDGEMMA_MODEL_NAME} (fp16, device_map='auto')...")
        tok = AutoTokenizer.from_pretrained(MEDGEMMA_MODEL_NAME)
        mg = AutoModelForCausalLM.from_pretrained(
            MEDGEMMA_MODEL_NAME,
            torch_dtype=torch.float16,
            device_map="auto",
        )
        mg.eval()
        if tok.pad_token_id is None:
            tok.pad_token_id = tok.eos_token_id

        new_rows = []
        t0 = time.time()
        for i, row in enumerate(tqdm(todo.itertuples(), total=len(todo), desc="MedGemma rewrite")):
            prompt_text = MEDGEMMA_REWRITE_TEMPLATE.format(sentence=row.sentence)
            messages = [{"role": "user", "content": prompt_text}]
            enc = tok.apply_chat_template(
                messages, return_tensors="pt", add_generation_prompt=True,
            ).to(mg.device)
            with torch.no_grad():
                out = mg.generate(
                    enc,
                    max_new_tokens=MEDGEMMA_MAX_NEW_TOK,
                    do_sample=False,
                    pad_token_id=tok.pad_token_id,
                )
            raw = tok.decode(out[0, enc.shape[1]:], skip_special_tokens=True)
            rewrite = _clean_rewrite_output(raw)
            if not rewrite:  # extremely rare — fall back to rule-strip
                rule_fallback = samples.loc[samples.sample_id == row.sample_id, "medsam3_prompt_rule"].iloc[0]
                rewrite = rule_fallback
            new_rows.append({"sample_id": row.sample_id, "medsam3_prompt_medgemma": rewrite})

            # Persist incrementally every 200 samples so a timeout doesn't lose work
            if (i + 1) % 200 == 0:
                interim = pd.concat([cached_df, pd.DataFrame(new_rows)], ignore_index=True)
                interim.to_csv(MEDGEMMA_REWRITE_CACHE, index=False)

        cached_df = pd.concat([cached_df, pd.DataFrame(new_rows)], ignore_index=True)
        cached_df.to_csv(MEDGEMMA_REWRITE_CACHE, index=False)
        print(f"\nWrote {len(cached_df)} total rewrites to {MEDGEMMA_REWRITE_CACHE}")
        print(f"Elapsed: {time.time()-t0:.1f}s for {len(new_rows)} new samples")

        # --- CRITICAL: free MedGemma from VRAM before loading SAM 3 -----
        del mg, tok
        gc.collect()
        torch.cuda.empty_cache()
        if torch.cuda.is_available():
            print(f"GPU memory after MedGemma unload: "
                  f"{torch.cuda.memory_allocated()/1e9:.2f} GB allocated, "
                  f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

    # --- merge rewrites into samples + set as active prompt -------------
    samples = samples.merge(cached_df, on="sample_id", how="left")
    # Fall back to rule prompt if MedGemma failed to produce one (e.g. cache miss)
    samples["medsam3_prompt"] = samples["medsam3_prompt_medgemma"].fillna(
        samples["medsam3_prompt_rule"]
    )

    n_used_medgemma = samples["medsam3_prompt_medgemma"].notna().sum()
    n_used_fallback = len(samples) - n_used_medgemma
    print(f"Active prompts: {n_used_medgemma} MedGemma, {n_used_fallback} rule fallback")

    # Quick side-by-side preview
    print("\nFirst 10 sentences — rule vs MedGemma:")
    preview = samples[["sentence","medsam3_prompt_rule","medsam3_prompt_medgemma"]].head(10)
    for _, r in preview.iterrows():
        print(f"  RAW : {r['sentence'][:90]}")
        print(f"  RULE: {r['medsam3_prompt_rule'][:90]}")
        print(f"  MGEM: {r['medsam3_prompt_medgemma'][:90] if pd.notna(r['medsam3_prompt_medgemma']) else '(no rewrite)'}")
        print()


## Cell 5 — GCS streaming image loader

No local copy. Each call opens an HTTPS stream of one PNG into RAM, hands
the bytes to PIL, and returns a `PIL.Image.Image`. After processing the
image goes out of scope and is freed — peak disk usage stays at zero.


In [ ]:
import io
from PIL import Image
import gcsfs

fs = gcsfs.GCSFileSystem(project=GCS_PROJECT)

def load_image_gcs(image_id: str) -> Image.Image:
    path = f"{BUCKET}/{IMG_PREFIX}/{image_id}"
    with fs.open(path, "rb") as f:
        data = f.read()
    return Image.open(io.BytesIO(data)).convert("RGB")

# Quick sanity check on the first sample (will raise if image not in bucket)
_test_id = samples.iloc[0]["image_id"]
_img = load_image_gcs(_test_id)
print(f"OK — sample image {_test_id}: size={_img.size}")


## Cell 6 — load MedSAM 3 (LoRA on facebook/sam3)

We use the helper functions from the official MedSAM 3 repo:
* `load_model_with_lora(base_model_name, lora_weights_path, device)` →
  `(Sam3Model, Sam3Processor)`.
* `segment_image(model, processor, PIL_image, text_prompt=...)` → np.ndarray
  of shape `(N, H, W)` of sigmoid probability maps.

If `LORA_WEIGHTS_FILE` doesn't exist in the LoRA repo under that name, list
the files at `https://huggingface.co/{LORA_REPO}/tree/main` and update the
filename in Cell 3.


In [ ]:
import sys, os
sys.path.insert(0, "/content/medsam3")

from huggingface_hub import hf_hub_download

# Pull the LoRA weights file from the public HF repo into the local Colab cache
lora_path = hf_hub_download(repo_id=LORA_REPO, filename=LORA_WEIGHTS_FILE)
print(f"LoRA weights at: {lora_path}")

# Import the inference helpers shipped in the MedSAM 3 repo
from inference import load_model_with_lora, segment_image  # type: ignore

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

model, processor = load_model_with_lora(
    base_model_name=BASE_MODEL,
    lora_weights_path=lora_path,
    device=DEVICE,
)
print("Model + processor loaded.")


## Cell 7 — IoU helpers

These mirror `project/src/compute_iou.py` so the notebook is fully
self-contained on Colab (Drive paths can be slow for many small imports).
If you edit one set, mirror the edits in the other.


In [ ]:
import numpy as np
from itertools import combinations
from typing import List

def boxes_to_mask(boxes, grid_size=MASK_GRID):
    mask = np.zeros((grid_size, grid_size), dtype=np.uint8)
    if not boxes: return mask
    for box in boxes:
        x1, y1, x2, y2 = box
        px1 = max(0, min(grid_size, int(np.floor(x1*grid_size))))
        py1 = max(0, min(grid_size, int(np.floor(y1*grid_size))))
        px2 = max(0, min(grid_size, int(np.ceil (x2*grid_size))))
        py2 = max(0, min(grid_size, int(np.ceil (y2*grid_size))))
        if px2 > px1 and py2 > py1:
            mask[py1:py2, px1:px2] = 1
    return mask

def mask_iou(m1, m2):
    inter = int(np.logical_and(m1, m2).sum())
    union = int(np.logical_or (m1, m2).sum())
    return inter/union if union > 0 else float("nan")

def boxes_to_intersection_mask(boxes, grid_size=MASK_GRID):
    if not boxes: return np.zeros((grid_size, grid_size), np.uint8)
    masks = [boxes_to_mask([b], grid_size) for b in boxes]
    out = masks[0].copy()
    for m in masks[1:]:
        out = np.logical_and(out, m).astype(np.uint8)
    return out

def boxes_to_outer_bbox_mask(boxes, grid_size=MASK_GRID):
    if not boxes: return np.zeros((grid_size, grid_size), np.uint8)
    xs1 = [b[0] for b in boxes]; ys1 = [b[1] for b in boxes]
    xs2 = [b[2] for b in boxes]; ys2 = [b[3] for b in boxes]
    return boxes_to_mask([[min(xs1), min(ys1), max(xs2), max(ys2)]], grid_size)

def pairwise_box_ious(boxes, grid_size=MASK_GRID):
    if not boxes or len(boxes) < 2: return []
    masks = [boxes_to_mask([b], grid_size) for b in boxes]
    return [mask_iou(masks[i], masks[j]) for i,j in combinations(range(len(masks)), 2)]

def pred_probs_to_grid_mask(probs, grid_size=MASK_GRID, threshold=MASK_THRESHOLD):
    arr = np.asarray(probs)
    if arr.ndim == 4 and arr.shape[0] == 1: arr = arr[0]
    if arr.ndim == 3: arr = arr.max(axis=0)
    if arr.ndim != 2: raise ValueError(f"Bad shape {probs.shape}")
    try:
        from skimage.transform import resize
        resized = resize(arr.astype(np.float32), (grid_size, grid_size),
                         order=1, mode="edge", anti_aliasing=False,
                         preserve_range=True)
    except Exception:
        h, w = arr.shape
        ys = np.linspace(0, h-1, grid_size).astype(np.int64)
        xs = np.linspace(0, w-1, grid_size).astype(np.int64)
        resized = arr[ys[:, None], xs[None, :]]
    return (resized > threshold).astype(np.uint8)

def compute_iou_bundle(pred_probs, r1_boxes, r2_boxes,
                       grid_size=MASK_GRID, threshold=MASK_THRESHOLD):
    pm = pred_probs_to_grid_mask(pred_probs, grid_size, threshold)
    all_boxes = (r1_boxes or []) + (r2_boxes or [])
    per_box = [mask_iou(pm, boxes_to_mask([b], grid_size)) for b in all_boxes]
    return {
        "per_box_ious":            per_box,
        "max_per_box_iou":         float(np.max(per_box))  if per_box else float("nan"),
        "mean_per_box_iou":        float(np.mean(per_box)) if per_box else float("nan"),
        "iou_with_pixel_or_union": mask_iou(pm, boxes_to_mask(all_boxes, grid_size)),
        "iou_with_outer_bbox_union": mask_iou(pm, boxes_to_outer_bbox_mask(all_boxes, grid_size)),
        "iou_with_intersection":   mask_iou(pm, boxes_to_intersection_mask(all_boxes, grid_size)),
        "iou_with_reader1_union":  mask_iou(pm, boxes_to_mask(r1_boxes or [], grid_size)),
        "iou_with_reader2_union":  mask_iou(pm, boxes_to_mask(r2_boxes or [], grid_size)),
        "reader_iou_union":        mask_iou(boxes_to_mask(r1_boxes or [], grid_size),
                                            boxes_to_mask(r2_boxes or [], grid_size)),
        "pairwise_box_ious":       pairwise_box_ious(all_boxes, grid_size),
        "mean_pairwise_iou":       float(np.mean(pairwise_box_ious(all_boxes, grid_size)))
                                   if len(all_boxes) >= 2 else float("nan"),
        "pred_area":               int(pm.sum()),
        "num_boxes_total":         len(all_boxes),
        "num_reader1_boxes":       len(r1_boxes or []),
        "num_reader2_boxes":       len(r2_boxes or []),
    }, pm


## Cell 8 — SMOKE TEST (50 samples, ~1–2 min on T4)

Randomly samples `SMOKE_N` rows stratified 25/25 by rule-based
uncertainty label (or all of one class if there aren't 25 of the other),
runs end-to-end inference + IoU, prints the first 5 IoU rows, and saves a
5×4 figure (image | GT boxes | predicted mask | overlay) so you can sanity
check both the segmentation and the prompt cleaning before committing to
the full ~13 K run.

If anything is wrong (auth, bucket path, prompt format, mask resolution)
this is where you catch it cheaply. Iterate on Cells 3–7 until the smoke
plot looks reasonable, **then** run Cell 9.

In [ ]:
import random, json
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(SMOKE_SEED)

def stratified_sample(df, n_per_class):
    out = []
    for cls in ["certain", "uncertain"]:
        sub = df[df["uncertainty_label_rule"] == cls]
        out.append(sub.sample(n=min(n_per_class, len(sub)), random_state=SMOKE_SEED))
    return pd.concat(out).sample(frac=1, random_state=SMOKE_SEED).reset_index(drop=True)

smoke = stratified_sample(samples, SMOKE_N // 2)
print(f"Smoke set: {len(smoke)} samples "
      f"({(smoke.uncertainty_label_rule=='uncertain').sum()} uncertain, "
      f"{(smoke.uncertainty_label_rule=='certain').sum()} certain)")

smoke_rows = []
sample_masks = {}     # sample_id -> mask (kept for the 5x4 figure)
sample_probs = {}

for row in tqdm(smoke.itertuples(), total=len(smoke), desc="smoke inference"):
    img = load_image_gcs(row.image_id)
    probs = segment_image(model, processor, img,
                          text_prompt=row.medsam3_prompt, device=DEVICE)
    r1 = json.loads(row.reader1_boxes) if isinstance(row.reader1_boxes, str) else row.reader1_boxes
    r2 = json.loads(row.reader2_boxes) if isinstance(row.reader2_boxes, str) else row.reader2_boxes
    bundle, pm = compute_iou_bundle(probs, r1, r2)
    rec = {
        "sample_id": row.sample_id,
        "image_id":  row.image_id,
        "finding_label": row.finding_label,
        "uncertainty_label_rule":     row.uncertainty_label_rule,
        "uncertainty_label_medgemma": row.uncertainty_label_medgemma,
        "medsam3_prompt": row.medsam3_prompt,
        "sentence_raw":   row.sentence,
        **bundle,
    }
    smoke_rows.append(rec)
    sample_masks[row.sample_id] = pm
    sample_probs[row.sample_id] = np.asarray(probs)

smoke_df = pd.DataFrame(smoke_rows)
print("\nFirst 5 IoU rows:")
display(smoke_df[[
    "sample_id","uncertainty_label_rule","medsam3_prompt",
    "iou_with_pixel_or_union","iou_with_outer_bbox_union",
    "mean_per_box_iou","reader_iou_union","num_boxes_total",
]].head(5))

# --- 5×4 sanity-check figure ---------------------------------------------
fig, axes = plt.subplots(5, 4, figsize=(16, 20))
for i, row in enumerate(smoke_df.head(5).itertuples()):
    img = load_image_gcs(row.image_id)
    iw, ih = img.size
    r1 = json.loads(samples.loc[samples.sample_id==row.sample_id, "reader1_boxes"].iloc[0])
    r2 = json.loads(samples.loc[samples.sample_id==row.sample_id, "reader2_boxes"].iloc[0])
    pm = sample_masks[row.sample_id]

    # 1. raw image
    axes[i,0].imshow(img, cmap="gray"); axes[i,0].set_title("image"); axes[i,0].axis("off")

    # 2. image + GT boxes
    axes[i,1].imshow(img, cmap="gray")
    for b, c in [(r1, "red"), (r2, "blue")]:
        for x1,y1,x2,y2 in b:
            axes[i,1].add_patch(patches.Rectangle(
                (x1*iw, y1*ih), (x2-x1)*iw, (y2-y1)*ih,
                lw=2, edgecolor=c, facecolor="none"))
    axes[i,1].set_title("GT boxes (R1=red, R2=blue)"); axes[i,1].axis("off")

    # 3. predicted mask
    axes[i,2].imshow(pm, cmap="gray"); axes[i,2].set_title(
        f"pred mask  IoU(or)={row.iou_with_pixel_or_union:.2f}"); axes[i,2].axis("off")

    # 4. overlay
    axes[i,3].imshow(img, cmap="gray")
    # resize predicted mask to image size for overlay (nearest-neighbor)
    pm_disp = np.array(Image.fromarray((pm*255).astype(np.uint8)).resize((iw, ih), Image.NEAREST))
    axes[i,3].imshow(np.ma.masked_where(pm_disp == 0, pm_disp),
                     cmap="Reds", alpha=0.5)
    axes[i,3].set_title(f"overlay [{row.uncertainty_label_rule}]\n{row.medsam3_prompt[:60]}")
    axes[i,3].axis("off")

plt.tight_layout()
plt.savefig("/content/medsam3_smoke_grid.png", dpi=110)
plt.show()
print("Saved smoke figure: /content/medsam3_smoke_grid.png")


## Cell 9 — FULL RUN (resumable)

Iterates the **full** sample set. For each row:

1. checks `gs://${BUCKET}/${OUT_PREFIX}/{sample_id}.json` — if present, skip
   (session-timeout-safe restart pattern),
2. otherwise streams the image, runs `segment_image`, computes the IoU bundle,
   uploads the IoU JSON and the predicted mask `.npz` to GCS.

Expected wall-clock:
* Colab T4 fp32: ~1–2 s per sample → ~3–4 h for ~13 K findings
* Colab A100 fp16: ~0.5 s per sample → ~1 h

If the session times out, re-run this cell — it picks up where it left
off.

In [ ]:
import io, json
import numpy as np
from tqdm.auto import tqdm

def exists_in_bucket(path):
    try:
        return fs.exists(path)
    except Exception:
        return False

def upload_json(path, obj):
    with fs.open(path, "w") as f:
        f.write(json.dumps(obj))

def upload_npz(path, mask):
    buf = io.BytesIO()
    np.savez_compressed(buf, mask=mask)
    buf.seek(0)
    with fs.open(path, "wb") as f:
        f.write(buf.read())

all_records = []
n_skipped = 0
n_new = 0

for row in tqdm(samples.itertuples(), total=len(samples), desc="full inference"):
    out_json = f"{BUCKET}/{OUT_PREFIX}/{row.sample_id}.json"
    out_npz  = f"{BUCKET}/{MASK_PREFIX}/{row.sample_id}.npz"
    if exists_in_bucket(out_json):
        with fs.open(out_json, "r") as f:
            all_records.append(json.loads(f.read()))
        n_skipped += 1
        continue
    try:
        img = load_image_gcs(row.image_id)
        probs = segment_image(model, processor, img,
                              text_prompt=row.medsam3_prompt, device=DEVICE)
        r1 = json.loads(row.reader1_boxes) if isinstance(row.reader1_boxes, str) else row.reader1_boxes
        r2 = json.loads(row.reader2_boxes) if isinstance(row.reader2_boxes, str) else row.reader2_boxes
        bundle, pm = compute_iou_bundle(probs, r1, r2)
        rec = {
            "sample_id": row.sample_id,
            "image_id":  row.image_id,
            "finding_label": row.finding_label,
            "uncertainty_label_rule":     row.uncertainty_label_rule,
            "uncertainty_label_medgemma": row.uncertainty_label_medgemma,
            "medsam3_prompt": row.medsam3_prompt,
            "sentence_raw":   row.sentence,
            **bundle,
        }
        upload_json(out_json, rec)
        upload_npz (out_npz,  pm)
        all_records.append(rec)
        n_new += 1
    except Exception as e:
        # Log + continue; we'd rather get 99% than crash on one bad image
        print(f"[skip] {row.sample_id}: {type(e).__name__}: {e}")
        continue

results_df = pd.DataFrame(all_records)
print(f"\nDone. Cached/skipped: {n_skipped}.  New this session: {n_new}.  "
      f"Total in results_df: {len(results_df)}.")
results_df.to_csv("/content/medsam3_per_sample_ious.csv", index=False)
print("Wrote /content/medsam3_per_sample_ious.csv")


## Cell 10 — aggregates stratified by rule-based uncertainty

* Group-level table — N, mean, median, std for each IoU variant by group.
* Per-finding × group table.
* Bootstrapped 95% CI for the headline metric (`iou_with_pixel_or_union` mean) per group.


In [ ]:
import json, numpy as np
RNG = np.random.default_rng(42)
BOOT_N = 2000

IOU_COLS = [
    "iou_with_pixel_or_union",
    "iou_with_outer_bbox_union",
    "iou_with_intersection",
    "iou_with_reader1_union",
    "iou_with_reader2_union",
    "reader_iou_union",
    "mean_per_box_iou",
    "max_per_box_iou",
    "mean_pairwise_iou",
]

# Coerce NaN-safe numeric
for c in IOU_COLS:
    results_df[c] = pd.to_numeric(results_df[c], errors="coerce")

# --- group-level table ----------------------------------------------------
group_agg = results_df.groupby("uncertainty_label_rule")[IOU_COLS].agg(
    ["count","mean","median","std"]
)
print(group_agg)

# --- per-finding × group table --------------------------------------------
pf = results_df.groupby(
    ["uncertainty_label_rule","finding_label"]
)[IOU_COLS].agg(["count","mean","median"])
pf.to_csv("/content/medsam3_per_finding_ious.csv")

# --- bootstrapped CI on iou_with_pixel_or_union mean ----------------------
def boot_mean_ci(x, n=BOOT_N, alpha=0.05):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]
    if len(x) == 0: return (float("nan"),)*3
    idx = RNG.integers(0, len(x), size=(n, len(x)))
    means = x[idx].mean(axis=1)
    lo, hi = np.quantile(means, [alpha/2, 1-alpha/2])
    return float(means.mean()), float(lo), float(hi)

agg_json = {}
for cls, sub in results_df.groupby("uncertainty_label_rule"):
    mean, lo, hi = boot_mean_ci(sub["iou_with_pixel_or_union"])
    agg_json[str(cls)] = {
        "n": int(len(sub)),
        "iou_with_pixel_or_union_mean": mean,
        "iou_with_pixel_or_union_ci95": [lo, hi],
        "iou_with_outer_bbox_union_mean": float(sub["iou_with_outer_bbox_union"].mean()),
        "iou_with_intersection_mean":     float(sub["iou_with_intersection"].mean()),
        "reader_iou_union_mean":          float(sub["reader_iou_union"].mean()),
        "mean_per_box_iou_mean":          float(sub["mean_per_box_iou"].mean()),
    }
with open("/content/medsam3_aggregate_ious.json", "w") as f:
    json.dump(agg_json, f, indent=2)
agg_json


## Cell 11 — plots

* Histogram of `iou_with_pixel_or_union` overlaid for certain vs uncertain.
* Per-group mean ± bootstrapped CI bar chart across all IoU variants.
* Scatter: annotator agreement (`reader_iou_union`) vs MedSAM-3
  performance (`iou_with_pixel_or_union`), colored by group — tests the
  "MedSAM 3 fails where annotators disagree" hypothesis.


In [ ]:
import matplotlib.pyplot as plt

# --- histogram ------------------------------------------------------------
plt.figure(figsize=(7,4))
for cls, color in [("certain","tab:blue"), ("uncertain","tab:orange")]:
    sub = results_df[results_df["uncertainty_label_rule"]==cls]["iou_with_pixel_or_union"].dropna()
    plt.hist(sub, bins=40, alpha=0.55, label=f"{cls} (n={len(sub)})", color=color)
plt.xlabel("IoU(pred, pixel-OR union of GT boxes)")
plt.ylabel("Count")
plt.title("MedSAM 3 IoU vs PadChest-GR, stratified by rule-based spatial uncertainty")
plt.legend()
plt.tight_layout()
plt.savefig("/content/medsam3_iou_histogram_by_group.png", dpi=120)
plt.show()

# --- per-group bar chart across IoU variants ------------------------------
plot_cols = [
    "iou_with_pixel_or_union",
    "iou_with_outer_bbox_union",
    "iou_with_intersection",
    "mean_per_box_iou",
    "max_per_box_iou",
    "reader_iou_union",
]
means = results_df.groupby("uncertainty_label_rule")[plot_cols].mean()
fig, ax = plt.subplots(figsize=(10,4))
means.T.plot(kind="bar", ax=ax)
ax.set_ylabel("Mean IoU")
ax.set_title("Mean IoU per metric, by rule-based uncertainty group")
ax.set_xticklabels(plot_cols, rotation=30, ha="right")
plt.tight_layout()
plt.savefig("/content/medsam3_iou_by_uncertainty_group.png", dpi=120)
plt.show()

# --- scatter: reader agreement vs MedSAM-3 IoU ----------------------------
plt.figure(figsize=(6,5))
for cls, color in [("certain","tab:blue"), ("uncertain","tab:orange")]:
    sub = results_df[results_df["uncertainty_label_rule"]==cls]
    plt.scatter(sub["reader_iou_union"], sub["iou_with_pixel_or_union"],
                s=10, alpha=0.5, label=cls, color=color)
plt.xlabel("Reader-Reader IoU (annotator agreement)")
plt.ylabel("MedSAM 3 IoU (pred vs union of GT boxes)")
plt.title("MedSAM 3 quality vs annotator agreement")
plt.legend()
plt.tight_layout()
plt.savefig("/content/medsam3_agreement_vs_iou_scatter.png", dpi=120)
plt.show()


## Cell 12 — push aggregated results back to the repo via Drive

Copies the final small artifacts (CSVs + JSON + 3 PNGs) into the
mirrored repo on Drive at `${DRIVE_REPO_ROOT}/project/...` so they appear
under `project/outputs/` and `project/figures/` after a `git pull` /
rsync on your laptop.

Per-sample masks and IoU JSONs **stay in GCS** (they're too many small
files to be ergonomic in git).


In [ ]:
import shutil, os

# Fall back to /content/ if Drive isn't mounted (DRIVE_MOUNTED set in Cell 4)
if globals().get("DRIVE_MOUNTED", False) and os.path.isdir("/content/drive/MyDrive"):
    out_dir = f"{DRIVE_REPO_ROOT}/project/outputs"
    fig_dir = f"{DRIVE_REPO_ROOT}/project/figures"
    print(f"Writing to Drive: {DRIVE_REPO_ROOT}")
else:
    out_dir = "/content/outputs"
    fig_dir = "/content/figures"
    print("Drive not mounted — writing to /content/. Use File → Download in Colab "
          "to grab the files, or run `from google.colab import files; "
          "files.download(...)`.")

os.makedirs(out_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

for fn in ("medsam3_per_sample_ious.csv",
           "medsam3_per_finding_ious.csv",
           "medsam3_aggregate_ious.json"):
    src = f"/content/{fn}"
    if os.path.exists(src):
        shutil.copy(src, f"{out_dir}/{fn}")

for fn in ("medsam3_iou_histogram_by_group.png",
           "medsam3_iou_by_uncertainty_group.png",
           "medsam3_agreement_vs_iou_scatter.png"):
    src = f"/content/{fn}"
    if os.path.exists(src):
        shutil.copy(src, f"{fig_dir}/{fn}")

print(f"Done. outputs → {out_dir}, figures → {fig_dir}")
